# Fine-tune Detection trên Colab — máy cá nhân không cần GPU
> Notebook tổng quát cũ để tham khảo. Với model/dữ liệu/script mới nhận và yêu cầu fine-tune tiếp từ C, dùng finetune_C_colab.ipynb cùng docs/chay-finetune-C-tren-colab.md và gói detection_C_colab_input.zip đã chuẩn bị.

Chỉ train detector 3 lớp `0 banana, 1 apple, 2 orange`. Không train classifier.

Đọc `docs/huong-dan-finetune-detection-colab.md` trước khi chạy. Trên Colab chọn runtime GPU.
Chuẩn bị Drive `MyDrive/fruit_detection/input/base.pt` và `dataset.zip`.
ZIP phải có `dataset/images/{train,val,test}` và `dataset/labels/{train,val,test}`.
`images/neg_val`, `images/neg_test` là tùy chọn, chỉ chứa ảnh đã xác nhận không có ba loại quả.
Dữ liệu train gồm dữ liệu cũ + ảnh thực tế đã kiểm tra nhãn; giữ nguyên test cũ.

Chạy từng ô từ trên xuống. Khi resume, làm theo phần 5; không chạy smoke hoặc train mới trước.
Notebook được kiểm tra cú pháp cục bộ; chưa chạy train GPU trong phiên tạo notebook.

## 1. Kiểm tra GPU và cài thư viện
Dừng ngay nếu không có GPU. Không cài toàn bộ requirements backend lên Colab.
Giữ Ultralytics cùng phiên bản repo; không tự nâng PyTorch/CUDA có sẵn trong runtime.
Nếu được yêu cầu restart runtime, restart rồi chạy lại từ đầu.

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa có GPU: chọn GPU runtime hoặc đợi quota; không train CPU."
print(torch.cuda.get_device_name(0))
%pip install -q ultralytics==8.4.180


## 2. Mount Drive, giải nén dữ liệu vào ổ cục bộ
Drive giữ file gốc và checkpoint; ảnh train đọc từ `/content` để tránh đọc nhiều file nhỏ trên Drive.
Sau khi runtime mất, chạy lại ô này để khôi phục dữ liệu. ZIP phải là bản cố định của thí nghiệm.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, zipfile
import yaml
import ultralytics
from ultralytics import YOLO

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/fruit_detection')
INPUT = DRIVE_ROOT / 'input'
LOCAL = Path('/content/fruit_detection')
LOCAL.mkdir(parents=True, exist_ok=True)
assert (INPUT / 'base.pt').is_file(), 'Thiếu input/base.pt từ lead.'
assert (INPUT / 'dataset.zip').is_file(), 'Thiếu input/dataset.zip.'
shutil.copy2(INPUT / 'base.pt', LOCAL / 'base.pt')
shutil.copy2(INPUT / 'dataset.zip', LOCAL / 'dataset.zip')
with zipfile.ZipFile(LOCAL / 'dataset.zip') as z:
    for entry in z.infolist():
        target = (LOCAL / entry.filename).resolve()
        assert target.is_relative_to(LOCAL.resolve()), 'ZIP có đường dẫn ngoài thư mục đích.'
    # Không tái sử dụng ảnh còn sót của một ZIP khác.
    if (LOCAL / 'dataset').exists():
        shutil.rmtree(LOCAL / 'dataset')
    z.extractall(LOCAL)
DATASET = LOCAL / 'dataset'
DATA_YAML = LOCAL / 'data.yaml'
NAMES = {0: 'banana', 1: 'apple', 2: 'orange'}
config = {'path': str(DATASET), 'train': 'images/train', 'val': 'images/val',
          'test': 'images/test', 'names': NAMES}
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

IDENTITY = {'dataset_zip_sha256': sha256(LOCAL / 'dataset.zip'),
            'base_pt_sha256': sha256(LOCAL / 'base.pt'),
            'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
            'gpu': torch.cuda.get_device_name(0)}
print(IDENTITY)
base = YOLO(str(LOCAL / 'base.pt'))
assert base.task == 'detect', f'Đây là model {base.task}, cần detector.'
assert base.names == NAMES, f'Sai mapping class của model: {base.names}'
del base


## 3. Kiểm tra ảnh, nhãn và trùng ảnh giữa các tập
Mỗi ảnh phải có file `.txt` cùng tên, kể cả ảnh nền (file rỗng).
Ô này phát hiện lỗi kỹ thuật và ảnh trùng byte; vẫn phải duyệt box bằng mắt và chia theo buổi chụp/video.
Ảnh gần giống nhau nhưng khác byte không được tự động phát hiện ở đây.
Không đưa nhãn 6 lớp trong `own_cam` trực tiếp vào dataset 3 lớp.

In [ ]:
from PIL import Image
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_files(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.suffix.lower() in EXTS and p.is_file())

seen = {}
report = {}
for split in ('train', 'val', 'test', 'neg_val', 'neg_test'):
    folder = DATASET / 'images' / split
    paths = image_files(folder)
    if split in ('train', 'val', 'test'):
        assert paths, f'Thiếu ảnh ở {folder}'
    counts = dict.fromkeys(NAMES, 0)
    negatives = 0
    for image in paths:
        with Image.open(image) as im:
            im.verify()
        digest = sha256(image)
        if digest in seen:
            old_split, old_path = seen[digest]
            assert old_split == split, f'Trùng ảnh giữa {old_split}/{split}: {old_path}, {image}'
        seen[digest] = (split, str(image))
        rel = image.relative_to(folder).with_suffix('.txt')
        label = DATASET / 'labels' / split / rel
        if split.startswith('neg_'):
            assert not label.exists() or not label.read_text().strip(), f'Ảnh nền có box: {label}'
            continue
        assert label.is_file(), f'Thiếu nhãn: {label}; ảnh nền cần file rỗng.'
        lines = [line for line in label.read_text().splitlines() if line.strip()]
        negatives += not lines
        for line in lines:
            fields = line.split()
            assert len(fields) == 5, f'Nhãn phải có 5 cột: {label}: {line}'
            c, x, y, w, h = map(float, fields)
            assert all(math.isfinite(v) for v in (c,x,y,w,h)), f'NaN/Inf: {label}'
            assert c.is_integer() and int(c) in NAMES, f'Sai class: {label}'
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, f'Sai tọa độ: {label}'
            assert x-w/2 >= -0.002 and y-h/2 >= -0.002 and x+w/2 <= 1.002 and y+h/2 <= 1.002, f'Box vượt ảnh: {label}'
            counts[int(c)] += 1
    report[split] = {'images': len(paths), 'boxes': counts, 'background_images': negatives}
    if split in ('train', 'val', 'test'):
        assert all(counts.values()), f'{split} thiếu box một loại quả: {counts}'
print(json.dumps(report, indent=2, ensure_ascii=False))


## 4. Chạy thử 1 epoch (chỉ khi train mới)
Xác nhận dữ liệu và VRAM ổn. Smoke có run riêng và không dùng làm điểm bắt đầu cho lượt chính.
Bỏ qua smoke khi resume bằng cách đặt RESUME=True trước khi chạy ô bên dưới.
Nếu OOM, restart runtime và đặt BATCH = 4 hoặc 2 trước khi thử lại.

In [ ]:
BATCH = 8
RUN_NAME = 'det_ft_cloud_v1'  # Đổi tên cho mỗi thí nghiệm mới.
PROJECT = LOCAL / 'runs'
BACKUP = DRIVE_ROOT / 'runs' / RUN_NAME
RESUME = False  # True chỉ khi tiếp tục lượt bị ngắt, giữ RUN_NAME cũ.

if not RESUME:
    smoke = YOLO(str(LOCAL / 'base.pt'))
    smoke.train(data=str(DATA_YAML), epochs=1, imgsz=640, batch=BATCH,
                device=0, workers=2, cache=False, amp=True,
                project=str(PROJECT), name='smoke', exist_ok=True,
                seed=42, optimizer='SGD', lr0=0.001, plots=False)
    del smoke
    import gc
    gc.collect()
    torch.cuda.empty_cache()


## 5. Fine-tune hoặc tiếp tục phiên bị ngắt
Lượt đầu: `RESUME=False`. Đã hoàn tất 1 epoch smoke thì chạy ô này.
Resume: chạy lại phần 1–3, đặt lại cấu hình phần 4 với `RESUME=True`, chạy ô cấu hình đó rồi ô này.
Checkpoint được sao lưu sau mỗi lần framework lưu model, thường cuối mỗi epoch.
Runtime chết giữa epoch sẽ mất phần đang chạy của epoch đó.
Không resume một lượt đã kết thúc; muốn train thêm thì tạo thí nghiệm mới từ trọng số đã chọn.

In [ ]:
RUN_DIR = PROJECT / RUN_NAME
if RESUME:
    assert (BACKUP / 'weights/last.pt').is_file(), 'Chưa có checkpoint trên Drive.'
    saved = json.loads((BACKUP / 'input_identity.json').read_text())
    for key in ('dataset_zip_sha256', 'base_pt_sha256', 'ultralytics'):
        assert saved[key] == IDENTITY[key], f'Khác {key}; không resume lẫn thí nghiệm.'
    shutil.copytree(BACKUP, RUN_DIR, dirs_exist_ok=True)
    model = YOLO(str(RUN_DIR / 'weights/last.pt'))
else:
    assert not RUN_DIR.exists() and not BACKUP.exists(), 'Run đã tồn tại; đổi RUN_NAME hoặc resume.'
    BACKUP.mkdir(parents=True)
    (BACKUP / 'input_identity.json').write_text(json.dumps(IDENTITY, indent=2), encoding='utf-8')
    model = YOLO(str(LOCAL / 'base.pt'))

def backup_run(trainer):
    # Copy qua file tạm để giảm nguy cơ ghi đè checkpoint bằng bản đang chép dở.
    for relative in ('weights/last.pt', 'weights/best.pt', 'args.yaml', 'results.csv'):
        src = Path(trainer.save_dir) / relative
        if src.is_file():
            dst = BACKUP / relative
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + '.tmp')
            shutil.copy2(src, tmp)
            os.replace(tmp, dst)

model.add_callback('on_model_save', backup_run)
if RESUME:
    model.train(resume=True, data=str(DATA_YAML), device=0, batch=BATCH, workers=2)
else:
    model.train(data=str(DATA_YAML), epochs=30, patience=8,
                imgsz=640, batch=BATCH, device=0, workers=2,
                optimizer='SGD', lr0=0.001, lrf=0.01,
                freeze=0, amp=True, cache=False, seed=42,
                close_mosaic=5, save=True,
                project=str(PROJECT), name=RUN_NAME, exist_ok=False)
backup_run(model.trainer)
# Sao lưu cả đồ thị, log và checkpoint cuối cùng sau khi train kết thúc.
shutil.copytree(Path(model.trainer.save_dir), BACKUP, dirs_exist_ok=True)
print('Đã lưu:', BACKUP)
del model
import gc
gc.collect()
torch.cuda.empty_cache()


## 6. So sánh trên validation — chọn checkpoint trước khi xem test
Đo lại baseline và cả best/last với cùng cấu hình. Không so thẳng với số mAP cũ ở conf 0.25.
Ở đây mAP dùng conf 0.001; ảnh nền dùng ngưỡng triển khai 0.40, NMS 0.60.
Nếu thiếu `neg_val`, cần kiểm tra thêm ảnh nền độc lập trước khi kết luận nhận nhầm đã giảm.
`best.pt` chọn theo validation fitness, không bảo đảm ít nhận nhầm nền nhất.

In [ ]:
import pandas as pd
EVAL_DIR = LOCAL / 'evaluation'
EVAL_DIR.mkdir(exist_ok=True)

def negative_rate(detector, split):
    paths = image_files(DATASET / 'images' / split)
    if not paths:
        return {'negative_images': 0, 'false_positive_images': None, 'fp_image_rate': None}
    bad = 0
    # Chia list thủ công: không phụ thuộc predict(list) có tôn trọng batch hay không.
    for start in range(0, len(paths), 8):
        for result in detector.predict([str(p) for p in paths[start:start+8]],
                stream=True, device=0, imgsz=640, conf=0.4, iou=0.6,
                agnostic_nms=True, quantize=16, verbose=False):
            bad += int(len(result.boxes) > 0)
    return {'negative_images': len(paths), 'false_positive_images': bad, 'fp_image_rate': bad/len(paths)}

def evaluate(weights, tag, split, negative_split):
    detector = YOLO(str(weights))
    assert detector.names == NAMES
    metrics = detector.val(data=str(DATA_YAML), split=split, imgsz=640,
                           batch=BATCH, device=0, workers=2, conf=0.001,
                           iou=0.6, agnostic_nms=True, quantize=16,
                           project=str(EVAL_DIR), name=tag, exist_ok=True, plots=True)
    row = {'model': tag, 'split': split, 'mAP50': float(metrics.box.map50),
           'mAP50_95': float(metrics.box.map)}
    for c, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        row[f'AP50_{NAMES[int(c)]}'] = float(ap)
    row.update(negative_rate(detector, negative_split))
    del detector
    torch.cuda.empty_cache()
    return row

CANDIDATES = {'baseline': LOCAL / 'base.pt',
              'best': BACKUP / 'weights/best.pt', 'last': BACKUP / 'weights/last.pt'}
val_rows = [evaluate(path, tag + '_val', 'val', 'neg_val') for tag, path in CANDIDATES.items()]
val_table = pd.DataFrame(val_rows)
val_table.to_csv(EVAL_DIR / 'validation_comparison.csv', index=False)
display(val_table)
shutil.copytree(EVAL_DIR, BACKUP / 'evaluation', dirs_exist_ok=True)


## 7. Chốt checkpoint rồi test một lần
Chọn `SELECTED` theo validation, AP từng loại, ảnh nền và duyệt lỗi.
Đề xuất thảo luận với lead: mAP50 giảm không quá 0.01, không mất nhiều recall chuối,
ảnh nhiều loại quả tốt hơn và tỷ lệ báo sai nền không tăng. Đây là tiêu chí đề xuất, không bảo đảm kết quả.
Nếu val chưa đủ ảnh thực tế/nền thì bổ sung val từ nhóm ảnh mới, không dùng test để dò tham số.

In [ ]:
SELECTED = 'best'  # Đổi thành 'last' hoặc 'baseline' sau khi xem validation.
assert SELECTED in CANDIDATES
selected_path = CANDIDATES[SELECTED]
final_rows = [evaluate(CANDIDATES['baseline'], 'baseline_test', 'test', 'neg_test')]
if SELECTED != 'baseline':
    final_rows.append(evaluate(selected_path, SELECTED + '_test', 'test', 'neg_test'))
final_table = pd.DataFrame(final_rows)
final_table.to_csv(EVAL_DIR / 'final_test_comparison.csv', index=False)
display(final_table)

# Vẽ một số ảnh test để duyệt box; không dùng chúng để tiếp tục chỉnh hyperparameter.
review = YOLO(str(selected_path))
review.predict([str(p) for p in image_files(DATASET / 'images/test')[:12]],
               device=0, imgsz=640, conf=0.4, iou=0.6, agnostic_nms=True,
               save=True, project=str(EVAL_DIR), name='preview', exist_ok=True)
shutil.copytree(EVAL_DIR, BACKUP / 'evaluation', dirs_exist_ok=True)
print('Xem ảnh kết quả ở:', EVAL_DIR / 'preview')


## 8. Xuất file để tải về
Tên file mới để thử trước khi thay model ứng dụng. Không ghi đè classifier.
Nếu chọn baseline thì file xuất chỉ là bản sao baseline, không được báo là model cải tiến.

In [ ]:
from google.colab import files
OUTPUT = DRIVE_ROOT / 'outputs' / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
exported = OUTPUT / 'det_candidate.pt'
shutil.copy2(selected_path, exported)
meta = dict(IDENTITY, run=RUN_NAME, selected=SELECTED, names=NAMES,
            inference={'conf': 0.4, 'iou': 0.6, 'imgsz': 640})
(OUTPUT / 'metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
shutil.copy2(EVAL_DIR / 'validation_comparison.csv', OUTPUT)
shutil.copy2(EVAL_DIR / 'final_test_comparison.csv', OUTPUT)
print('Model và báo cáo trên Drive:', OUTPUT)
files.download(str(exported))
